In [150]:
import duckdb
import pandas as pd
import os
from datetime import datetime

In [ ]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

: 

In [136]:
arquivo = 'z0019_01.csv'
data_ingestao = datetime.now()
df = pd.read_csv(f'../landing/{arquivo}', sep=';')
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestao 
df.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,Parafuso,BT10,100,100,z0019_01.csv,2025-10-21 18:45:25.320300
1,10002,Martelo,BT50,100,1500,z0019_01.csv,2025-10-21 18:45:25.320300
2,10003,Prego,BT10,100,50,z0019_01.csv,2025-10-21 18:45:25.320300
3,10004,Serrote,BT20,200,200,z0019_01.csv,2025-10-21 18:45:25.320300


In [137]:
con.execute("""
    CREATE TABLE IF NOT EXISTS bronze_produtos ( 
        NATBR VARCHAR,
        MAKTX VARCHAR,
        WERKS VARCHAR,
        MAINS VARCHAR,
        LABST VARCHAR,
        nome_arquivo VARCHAR,
        data_ingestao TIMESTAMP
    )
""")

EXTRA - PARA CORRIGIR O PROBLEMA

In [138]:
# Garantir que a coluna MAINS exista logo após WERKS na tabela bronze_produtos
# Estratégia: se a tabela existe e MAINS não existe, cria-se uma tabela temporária com a ordem desejada, copia-se os dados, dropa-se a tabela antiga e renomeia-se a nova.
try:
    cols = [r[0] for r in con.execute("DESCRIBE bronze_produtos").fetchall()]
except Exception:
    # tabela não existe: criar com a ordem desejada
    con.execute("""
        CREATE TABLE IF NOT EXISTS bronze_produtos (
            NATBR VARCHAR,
            MAKTX VARCHAR,
            WERKS VARCHAR,
            MAINS VARCHAR,
            LABST VARCHAR,
            nome_arquivo VARCHAR,
            data_ingestao TIMESTAMP
        )
    """)
    print('Tabela bronze_produtos criada com coluna MAINS posicionada após WERKS')
else:
    # tabela existe
    if 'MAINS' not in cols:
        # criar tabela temporária com a ordem desejada
        con.execute("""
            CREATE TABLE bronze_produtos_new (
                NATBR VARCHAR,
                MAKTX VARCHAR,
                WERKS VARCHAR,
                MAINS VARCHAR,
                LABST VARCHAR,
                nome_arquivo VARCHAR,
                data_ingestao TIMESTAMP
            )
        """)
        # copiar dados: se a coluna MAINS não existe, inserir NULL para ela
        con.execute("""
            INSERT INTO bronze_produtos_new
            SELECT
                NATBR,
                MAKTX,
                WERKS,
                NULL AS MAINS,
                LABST,
                nome_arquivo,
                data_ingestao
            FROM bronze_produtos
        """)
        con.execute("DROP TABLE bronze_produtos")
        con.execute("ALTER TABLE bronze_produtos_new RENAME TO bronze_produtos")
        print('Coluna MAINS adicionada e posicionada após WERKS')
    else:
        print('Coluna MAINS já existe na tabela')

Coluna MAINS já existe na tabela


EXTRA - PARA CORRIGIR PROBLEMA

In [139]:
# Inserção segura garantindo persistência dos valores de MAINS
# 1) verificar que df tem MAINS
if 'MAINS' not in df.columns:
    raise ValueError("DataFrame não contém a coluna 'MAINS'")
# 2) garantir que a tabela possui a coluna MAINS
cols = [r[0] for r in con.execute("DESCRIBE bronze_produtos").fetchall()]
if 'MAINS' not in cols:
    con.execute("ALTER TABLE bronze_produtos ADD COLUMN MAINS VARCHAR")
# 3) remover linhas previamente ingeridas deste arquivo para evitar duplicatas
con.execute(f"DELETE FROM bronze_produtos WHERE nome_arquivo = '{arquivo}'")
# 4) registrar e inserir explicitamente incluindo MAINS
con.register('df', df)
con.execute("""
    INSERT INTO bronze_produtos (NATBR, MAKTX, WERKS, MAINS, LABST, nome_arquivo, data_ingestao)
    SELECT NATBR, MAKTX, WERKS, MAINS, LABST, nome_arquivo, data_ingestao FROM df
""")

In [140]:
resultado = con.execute("SELECT * FROM bronze_produtos").fetchdf()
resultado.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,Parafuso,BT10,100,100,z0019_01.csv,2025-10-21 18:45:25.320300
1,10002,Martelo,BT50,100,1500,z0019_01.csv,2025-10-21 18:45:25.320300
2,10003,Prego,BT10,100,50,z0019_01.csv,2025-10-21 18:45:25.320300
3,10004,Serrote,BT20,200,200,z0019_01.csv,2025-10-21 18:45:25.320300


VER O PROXIMO ARQUIVO z0019_02.csv

In [141]:
arquivo = 'z0019_02.csv'
data_ingestao = datetime.now()
df = pd.read_csv(f'../landing/{arquivo}', sep=';')
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestao 
df.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10003,Prego,BT10,100,60,z0019_02.csv,2025-10-21 18:45:26.082588
1,10004,Serra,BT20,200,200,z0019_02.csv,2025-10-21 18:45:26.082588
2,10005,Alicate,BT50,100,300,z0019_02.csv,2025-10-21 18:45:26.082588
3,10006,Chave de fenda,BT10,100,400,z0019_02.csv,2025-10-21 18:45:26.082588
4,10007,Furadeira,BT30,150,75,z0019_02.csv,2025-10-21 18:45:26.082588


In [142]:
con.execute("""
    CREATE TABLE IF NOT EXISTS bronze_produtos ( 
        NATBR VARCHAR,
        MAKTX VARCHAR,
        WERKS VARCHAR,
        MAINS VARCHAR,
        LABST VARCHAR,
        nome_arquivo VARCHAR,
        data_ingestao TIMESTAMP
    )
""")

In [148]:
resultado = con.execute("SELECT * FROM bronze_z0019").fetchdf()
resultado.head(6)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10003,Prego,BT10,100,60,z0019_02.csv,2025-10-21 17:47:31.029225
1,10004,Serra,BT20,200,200,z0019_02.csv,2025-10-21 17:47:31.029225
2,10005,Alicate,BT50,100,300,z0019_02.csv,2025-10-21 17:47:31.029225
3,10006,Chave de fenda,BT10,100,400,z0019_02.csv,2025-10-21 17:47:31.029225
4,10007,Furadeira,BT30,150,75,z0019_02.csv,2025-10-21 17:47:31.029225
5,10008,Lixadeira,BT20,200,125,z0019_02.csv,2025-10-21 17:47:31.029225


In [144]:
con.execute("INSERT INTO bronze_produtos SELECT * FROM df")

In [147]:
con.execute("ALTER TABLE bronze_produtos RENAME TO bronze_z0019")

CatalogException: Catalog Error: Could not rename "bronze_produtos" to "bronze_z0019": another entry with this name already exists!

In [ ]:
con.close()